# 🛡️ Lead.AI Fraud Shield — Explainable Fraud Detection

**Building, evaluating, and explaining a fraud risk classification model with SHAP**

> Published by [Arun Kumar Gharami](https://www.kaggle.com/arungharami) · [Lead.AI Labs](https://www.lead-ai.us)

---

## Business Problem

Financial fraud costs businesses 3–5× the face value of each fraudulent transaction once
chargebacks, dispute fees, and manual review time are factored in. Most small businesses
have no automated fraud detection — they rely on manual spot-checks or simple rule engines
that produce high false-positive rates and no explanation.

This notebook demonstrates how to build an **explainable fraud detection model** that:
- Classifies transactions as Low / Medium / High risk
- Uses SHAP to explain every prediction in plain English
- Gives analysts a reason for each flag — not just a score

**Links:**
- 🤗 [Hugging Face Model](https://huggingface.co/arun-gharami/lead-ai-fraud-shield)
- 🖥️ [Live Demo](https://huggingface.co/spaces/arun-gharami/fraud-detection-xai-demo)
- 💻 [GitHub](https://github.com/Arungharami/lead-ai-fraud-shield)
- 🌐 [Lead.AI Labs](https://www.lead-ai.us)

## 1. Setup & Imports

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install shap matplotlib seaborn scikit-learn pandas numpy -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from sklearn.preprocessing import label_binarize
import shap

shap.initjs()
print('All imports successful ✓')

## 2. Load Dataset

In [ ]:
# Load from Kaggle dataset
df = pd.read_csv('/kaggle/input/lead-ai-fraud-detection/train.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
df.head()

In [ ]:
print('Class distribution:')
print(df['risk_label'].value_counts())
print(f'\nFraud rate: {df["risk_label"].mean():.2%}')
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Lead.AI Fraud Detection — EDA', fontsize=16, fontweight='bold')

# Fraud rate by transaction hour
hourly = df.groupby('transaction_hour')['risk_label'].mean()
axes[0,0].bar(hourly.index, hourly.values, color=['#dc2626' if v > 0.15 else '#16a34a' for v in hourly.values])
axes[0,0].set_title('Fraud Rate by Hour')
axes[0,0].set_xlabel('Hour of Day')
axes[0,0].set_ylabel('Fraud Rate')

# Transaction amount distribution
df[df['risk_label']==0]['transaction_amount'].hist(bins=50, alpha=0.6, label='Normal', ax=axes[0,1], color='#16a34a')
df[df['risk_label']==1]['transaction_amount'].hist(bins=50, alpha=0.6, label='Fraud', ax=axes[0,1], color='#dc2626')
axes[0,1].set_title('Transaction Amount Distribution')
axes[0,1].legend()
axes[0,1].set_xlim(0, 3000)

# Fraud rate by merchant category
cat_fraud = df.groupby('merchant_category')['risk_label'].mean().sort_values(ascending=True)
cat_fraud.plot(kind='barh', ax=axes[0,2], color='#f59e0b')
axes[0,2].set_title('Fraud Rate by Merchant Category')
axes[0,2].set_xlabel('Fraud Rate')

# Account age vs fraud
axes[1,0].scatter(
    df[df['risk_label']==0]['account_age_days'].sample(min(1000, len(df))),
    df[df['risk_label']==0]['transaction_amount'].sample(min(1000, len(df))),
    alpha=0.3, label='Normal', color='#16a34a', s=5
)
axes[1,0].scatter(
    df[df['risk_label']==1]['account_age_days'],
    df[df['risk_label']==1]['transaction_amount'],
    alpha=0.5, label='Fraud', color='#dc2626', s=10
)
axes[1,0].set_title('Account Age vs Amount')
axes[1,0].set_xlabel('Account Age (days)')
axes[1,0].set_ylabel('Transaction Amount')
axes[1,0].legend()

# Fraud rate by device type
dev_fraud = df.groupby('device_type')['risk_label'].mean().sort_values()
dev_fraud.plot(kind='bar', ax=axes[1,1], color='#6366f1', rot=0)
axes[1,1].set_title('Fraud Rate by Device Type')
axes[1,1].set_ylabel('Fraud Rate')

# Transaction velocity
vel_fraud = df.groupby('transaction_velocity_1h')['risk_label'].mean()
axes[1,2].plot(vel_fraud.index[:20], vel_fraud.values[:20], 'o-', color='#dc2626')
axes[1,2].set_title('Fraud Rate by 1h Velocity')
axes[1,2].set_xlabel('Transactions in Last Hour')
axes[1,2].set_ylabel('Fraud Rate')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Engineering & Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_model = df.copy()

# Encode categoricals
for col in ['merchant_category', 'transaction_country', 'device_type',
            'transaction_type', 'geo_location_region']:
    if col in df_model.columns:
        le = LabelEncoder()
        df_model[col + '_enc'] = le.fit_transform(df_model[col].fillna('unknown'))

# Select numeric + encoded features
FEATURE_COLS = [
    'transaction_amount', 'transaction_hour', 'transaction_day_of_week',
    'is_weekend', 'account_age_days', 'previous_chargebacks',
    'is_international', 'is_high_risk_merchant_category',
    'customer_total_transactions_30d', 'customer_risk_score',
    'avg_transaction_amount_30d_customer',
    'transaction_velocity_1h', 'transaction_velocity_24h',
    'merchant_category_enc', 'device_type_enc', 'transaction_type_enc'
]

# Use only columns that exist in this dataset
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_model.columns]

X = df_model[FEATURE_COLS].fillna(0)
y = df_model['risk_label']

print(f'Features used: {len(FEATURE_COLS)}')
print(FEATURE_COLS)

## 5. Train Baseline Model (Random Forest)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

baseline = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

print('=== Baseline Model (Random Forest) ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_base):.4f}')
print(classification_report(y_test, y_pred_base))

## 6. Train Improved Model (Gradient Boosting)

In [ ]:
gbm = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, random_state=42
)
gbm.fit(X_train, y_train)
y_pred_gbm = gbm.predict(X_test)

print('=== Improved Model (Gradient Boosting) ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_gbm):.4f}')
print(classification_report(y_test, y_pred_gbm))

## 7. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, model, preds, title in [
    (axes[0], baseline, y_pred_base, 'Random Forest (Baseline)'),
    (axes[1], gbm,      y_pred_gbm,  'Gradient Boosting (Improved)'),
]:
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Fraud'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)

plt.suptitle('Confusion Matrices — Lead.AI Fraud Shield', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. ROC-AUC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for model, label, color in [
    (baseline, 'Random Forest', '#6366f1'),
    (gbm,      'Gradient Boosting', '#dc2626'),
]:
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{label} (AUC={auc:.3f})', color=color, lw=2)

ax.plot([0,1],[0,1],'k--', label='Random Baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC-AUC — Lead.AI Fraud Shield')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_auc.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Feature Importance

In [ ]:
importance = pd.Series(gbm.feature_importances_, index=FEATURE_COLS)
importance = importance.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importance.plot(kind='barh', color='#f59e0b')
plt.title('Feature Importance — Gradient Boosting', fontsize=14, fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 features:')
print(importance.tail(5))

## 10. SHAP Explainability

In [ ]:
print('Computing SHAP values (this may take ~30 seconds)...')
explainer = shap.TreeExplainer(gbm)

# Use a sample for speed
X_sample = X_test.sample(min(500, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_sample)
print('SHAP values computed ✓')

In [ ]:
# Summary plot — global feature importance via SHAP
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_sample, plot_type='bar', show=False)
plt.title('SHAP Feature Importance — Lead.AI Fraud Shield', fontsize=13)
plt.tight_layout()
plt.savefig('shap_summary_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Beeswarm plot — direction of impact
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_sample, show=False)
plt.title('SHAP Beeswarm — Feature Impact Direction', fontsize=13)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Single Transaction Explanation

In [ ]:
# Pick a high-risk transaction and explain it
fraud_samples = X_test[y_test == 1]

if len(fraud_samples) > 0:
    sample = fraud_samples.iloc[[0]]
    sv = explainer.shap_values(sample)

    print('=== Transaction Details ===')
    print(sample.T.to_string())
    print(f'\nModel Prediction: {gbm.predict(sample)[0]} (0=Normal, 1=Fraud)')
    print(f'Fraud Probability: {gbm.predict_proba(sample)[0][1]:.2%}')

    # Feature impacts for this prediction
    feature_impacts = pd.Series(sv[0], index=FEATURE_COLS).sort_values(ascending=False)
    print('\n=== SHAP Feature Impacts (top 5) ===')
    print(feature_impacts.head())

    # Waterfall plot
    shap.waterfall_plot(
        shap.Explanation(
            values=sv[0],
            base_values=explainer.expected_value,
            data=sample.values[0],
            feature_names=FEATURE_COLS,
        ),
        show=True
    )

## 12. Business Interpretation

In [ ]:
# Business-level summary
test_results = X_test.copy()
test_results['actual']     = y_test.values
test_results['predicted']  = y_pred_gbm
test_results['fraud_prob'] = gbm.predict_proba(X_test)[:, 1]

# Risk tier distribution
test_results['risk_tier'] = pd.cut(
    test_results['fraud_prob'],
    bins=[-0.001, 0.35, 0.65, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

print('=== Risk Tier Distribution ===')
print(test_results['risk_tier'].value_counts())

print('\n=== Fraud Capture Rate by Tier ===')
for tier in ['Low Risk', 'Medium Risk', 'High Risk']:
    tier_df = test_results[test_results['risk_tier'] == tier]
    if len(tier_df) > 0:
        fraud_rate = tier_df['actual'].mean()
        print(f'{tier:15s}: {len(tier_df):5d} transactions | {fraud_rate:.1%} actual fraud rate')

In [ ]:
# Estimated business value
avg_transaction = test_results['transaction_amount'].mean() if 'transaction_amount' in test_results.columns else 250
total_fraud = test_results['actual'].sum()
detected_fraud = ((test_results['predicted'] == 1) & (test_results['actual'] == 1)).sum()
detection_rate = detected_fraud / total_fraud if total_fraud > 0 else 0

print(f'\n=== Business Value Estimate ===')
print(f'Total fraud transactions in test set: {total_fraud}')
print(f'Detected by model:                    {detected_fraud} ({detection_rate:.1%})')
print(f'Average transaction amount:           ${avg_transaction:.0f}')
print(f'Estimated fraud prevented (3x cost):  ${detected_fraud * avg_transaction * 3:,.0f}')
print(f'\nAt this detection rate, a business processing $500K/month could save')
print(f'an estimated ${500000 * 0.005 * detection_rate * 3:,.0f}/month in fraud costs.')

## Summary

In this notebook we:
1. Loaded the Lead.AI synthetic fraud detection dataset (100K transactions, 21 features)
2. Performed exploratory analysis — identifying key fraud signals: late-night transactions,
   new accounts, high-risk merchants, and elevated device/location risk scores
3. Trained a baseline Random Forest and an improved Gradient Boosting classifier
4. Evaluated both models with accuracy, classification report, confusion matrix, and ROC-AUC
5. Used SHAP TreeExplainer to identify global and per-transaction feature importance
6. Translated model predictions into business-level risk tiers and estimated ROI

**Key findings:**
- `customer_risk_score`, `transaction_velocity_1h`, and `account_age_days` are the strongest
  fraud predictors in this synthetic dataset
- Transactions flagged as High Risk have a substantially higher actual fraud rate
- SHAP explanations make each prediction auditable and analyst-ready

---

## Next Steps

- 🤗 **Try the live model:** [Fraud Shield on Hugging Face](https://huggingface.co/arun-gharami/lead-ai-fraud-shield)
- 🖥️ **Interactive demo:** [XAI Demo Space](https://huggingface.co/spaces/arun-gharami/fraud-detection-xai-demo)
- 💻 **Full source code:** [GitHub](https://github.com/Arungharami/lead-ai-fraud-shield)
- 🚀 **Deploy for your business:** [lead-ai.us](https://www.lead-ai.us)
- 💼 **Connect:** [LinkedIn — Arun Kumar Gharami](https://www.linkedin.com/in/arunkgharami)

> ⚠️ All data in this notebook is **synthetic**. Models are for educational and research use only.